# 06 · Split Strategy & Leakage Tests

**Amaç:** Time-based split kararını doğrulamak; train/test entity overlap'ını ölçmek.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated

from src.models.split import time_based_split, random_split, entity_overlap

In [ ]:
df, _ = load_validated()
from src.features.instant import add_derived
df = add_derived(df)

## Time-based split

In [ ]:
split = time_based_split(df)
print('Train:', len(split.train), 'rows  |  fraud rate:', f'{split.train["IsFraudTransaction"].mean()*100:.3f}%')
print('Val:  ', len(split.val), '  |  fraud rate:', f'{split.val["IsFraudTransaction"].mean()*100:.3f}%')
print('Test: ', len(split.test), '  |  fraud rate:', f'{split.test["IsFraudTransaction"].mean()*100:.3f}%')
print('Cutoffs:', split.cutoffs)

## Entity overlap (train ↔ test)

In [ ]:
overlap = entity_overlap(split.train, split.test)
for col, info in overlap.items():
    print(col, info)

## Random split benchmark

In [ ]:
rs = random_split(df)
print('Random Test fraud rate:', f'{rs.test["IsFraudTransaction"].mean()*100:.3f}%')

## Sonuç
- Time-based split: train fraud rate ~%0.9, test fraud rate ~%0.2 (drift gerçek).
- Train ↔ test arasında DeviceId ve ReceiverName overlap'ı yüksek; bu nedenle ham entity ID asla modele verilmez.
- Random split benchmark olarak raporlanır; karar metrikleri time-split üzerinden okunur.